# Intermediate NN

In [3]:
#########################     LIBRARIES     ##########################
from keras.models import Model
from keras.layers import Dense, Input
#from keras.layers.merge import concatenate
from tensorflow.keras.layers import concatenate     # PROBLEMA SEMBRA RISOLTO COSI !!!
import keras.backend as K
from keras.regularizers import l2
from hyperopt import STATUS_OK, tpe, Trials, hp, fmin
from hyperopt.pyll.stochastic import sample
from sklearn.model_selection import KFold
import numpy as np
from matplotlib import pyplot as plt
from math import pi
from keras.optimizers import Adam,Nadam,Adamax
from ann_functions import getModel, kCrossValGP, transfBestparam
from time import perf_counter
import pandas
import pickle
import os

seed = 7
np.random.seed(seed)

In [4]:
########################     PREPARATION      ##########################
HF_data = np.loadtxt("../DATA_shear_cube/Data_Train_bases_young/HF_num_data.txt").astype(int) #list of number of HF data
Nlf_models = np.loadtxt("../DATA_shear_cube/Data_Train_bases_young/N_bases.txt").astype(int)[0:2]   #list of number of bases
HF_data_str = [str(num) for num in HF_data]
r2_df = pandas.DataFrame(index=HF_data_str)      #dataframe which stores overall R^2
r2_HF_df = pandas.DataFrame(index=HF_data_str)   #dataframe which stores HF R^2
r2_LF_df = pandas.DataFrame(index=HF_data_str)   #dataframe which stores LF R^2
mse_df = pandas.DataFrame(index=HF_data_str)     #dataframe which stores overall MSE
mse_HF_df = pandas.DataFrame(index=HF_data_str)  #dataframe which stores HF MSE
mse_LF_df = pandas.DataFrame(index=HF_data_str)  #dataframe which stores LF MSE
#mu_train_LF = np.loadtxt("../DATA_shear_cube/Data_Train_bases_lin_young/mu_train_LF_young.txt")[:,0]
#mu_test = np.loadtxt("../DATA_shear_cube/Data_Test_bases_lin_young/mu_test_young.txt")[:,0]
#U_lf_train_full = np.loadtxt("../DATA_shear_cube/Data_Train_bases_lin_young/Ulf_train_young.txt")
#U_lf_test_full = np.loadtxt("../DATA_shear_cube/Data_Test_bases_lin_young/Ulf_test_young.txt")
#U_hf_test = np.loadtxt("../DATA_shear_cube/Data_Test_bases_lin_young/Uhf_test_young.txt")


U_HF_list = []
U_LF_list = []

In [5]:
for m in range(len(Nlf_models)):
#Loop over the range of the number of basis functions

    print(f"********************  #basis functions = {Nlf_models[m]}  ********************")
    test_mse_HF_list = []
    test_mse_LF_list = []
    test_mse_list = []
    r2_HF_list = []
    r2_LF_list = []
    r2_list = []
    
    mu_train_LF = np.loadtxt("../DATA_shear_cube/Data_Train_bases_young/mu_train_LF_young.txt")[:,0]     # <-
    Nlf = np.size(mu_train_LF)     #number of low-fidelity data to TRAIN the NN
    U_lf_train_full = np.loadtxt("../DATA_shear_cube/Data_Train_bases_young/Ulf_train_young.txt")      # <-

    permutation = np.random.permutation(len(mu_train_LF))
    mu_train_LF=mu_train_LF[permutation][0:10]

    U_lf_train_full=U_lf_train_full[permutation][0:10]
    
    for n_HF in HF_data:
    #loop over the possible numbers of high-fidelity data

        print(f"-------  #HF data = {n_HF}  -------")
        start = perf_counter()

        n_HF_txt = str(n_HF) + '.txt'

        #########################     TRAIN SET      ##########################
        mu_train_HF = np.loadtxt("../DATA_shear_cube/Data_Train_bases_young/mu_train_HF_young_" + n_HF_txt)[:,0]

        Nhf = np.size(mu_train_HF)     #number of high-fidelity data to TRAIN the NN
        N = Nhf + Nlf

        U_hf_train = np.loadtxt("../DATA_shear_cube/Data_Train_bases_young/Uhf_train_young_" + n_HF_txt)      # ""

        U_lf_train = U_lf_train_full[:,m]

        Nepo = 3000  #number of epoches


        # reduction
        permutation = np.random.permutation(len(mu_train_HF))
        mu_train_HF=mu_train_HF[permutation][0:5]

        U_hf_train=U_hf_train[permutation][0:5]

        #########################     TEST SET      ##########################
        mu_test = np.loadtxt("../DATA_shear_cube/Data_Test_bases_young/mu_test_young.txt")[:,0]    # <-

        N_test = np.size(mu_test)

        U_lf_test_full = np.loadtxt("../DATA_shear_cube/Data_Test_bases_young/Ulf_test_young.txt")    # <-        AGGIUNGERE NOISE?
        U_hf_test = np.loadtxt("../DATA_shear_cube/Data_Test_bases_young/Uhf_test_young.txt")    #<-

        U_lf_test = U_lf_test_full[:,m]



        ########################     NORMALIZATION      ########################
        #Input
        mu_max = np.max(mu_test)
        mu_min = np.min(mu_test)

        mu_test_norm = (mu_test - mu_min) / (mu_max - mu_min)
        mu_train_LF_norm = (mu_train_LF - mu_min) / (mu_max - mu_min)
        mu_train_HF_norm = (mu_train_HF - mu_min) / (mu_max - mu_min)

        #Output
        hfmean = lfmean = np.mean(U_hf_test)

        U_lf_train = U_lf_train - lfmean
        U_hf_train = U_hf_train - hfmean
        U_lf_test = U_lf_test - lfmean
        U_hf_test = U_hf_test - hfmean

        mu_train_norm = np.concatenate((mu_train_HF_norm, mu_train_LF_norm))

        '''
        ##################     Print LF and HF models      ###################
        plt.figure()
        plt.plot(mu_test, J_hf_test, 'b-', linewidth = 1.5, label = 'HF model')
        plt.plot(mu_train_HF, J_hf_train,'b*', markersize = 7, label = 'HF training points')
        plt.plot(mu_test, J_lf_test,'r--', linewidth = 1.5, label = 'LF model')
        plt.plot(mu_train_LF, J_lf_train,'r*', markersize = 5, label = 'LF training points')
        plt.legend(loc = 1, prop={'size':8.3})
        '''


        ##########################     TRAINING      ##########################
        K.clear_session()

        name = 'Inter'
        MAX_EVAL = 30
        #best paramters obtained by HPO:
        best_params = {'alpha': 0.031884991755260814, 'epochs': 2.0, 'kernel_init': 'uniform', 'l2weight': 0.002651788904350721, 'lr': 0.00042698780348019073, 'nodes': 114.0, 'opt': 'Adamax'}
        #best_params = {'alpha': 0.04065167240033655, 'epochs': 3.0, 'kernel_init': 'uniform', 'l2weight': 0.00021154214219451555, 'lr': 0.0065338127394483905, 'nodes': 128.0, 'opt': 'Adamax'}

        finalModel = getModel(best_params,name)
        hist = finalModel.fit(mu_train_norm, np.concatenate((U_hf_train, U_lf_train)),validation_data=(mu_test_norm,[U_hf_test, U_lf_test]), epochs=Nepo * int(best_params['epochs']),batch_size=N, verbose = 0, validation_freq = 50)

        U_pred = finalModel.predict(mu_test_norm)

        stop = perf_counter()
        elapsed = stop - start
        print('Elapsed time: ', elapsed)

        U_Pred = np.concatenate((U_pred[0],U_pred[1]))
        U_HF_list.append(U_pred[0][:,0])
        U_LF_list.append(U_pred[1][:,0])
        U_test = np.concatenate((U_hf_test,U_lf_test))

        test_mse= np.mean(np.square(U_test- U_Pred[:,0]))
        test_mse_list.append(test_mse)
        print(f"Test MSE: {test_mse:.8f}")

        test_mse_HF = np.mean(np.square(U_hf_test- U_pred[0][:,0]))
        test_mse_HF_list.append(test_mse_HF)
        print(f"Test MSE HF: {test_mse_HF:.8f}")

        test_mse_LF = np.mean(np.square(U_lf_test- U_pred[1][:,0]))
        test_mse_LF_list.append(test_mse_LF)
        print(f"Test MSE LF: {test_mse_LF:.8f}")

        r2 = 1 - np.sum(np.square(U_test - U_Pred[:,0])) / np.sum(np.square(U_test - np.mean(U_test)))
        r2_list.append(r2)
        print(f"R^2: {r2:.4f}")

        r2_HF = 1 - np.sum(np.square(U_hf_test - U_pred[0][:,0])) / np.sum(np.square(U_hf_test - np.mean(U_hf_test)))
        r2_HF_list.append(r2_HF)
        print(f"R^2 HF: {r2_HF:.4f}")

        r2_LF = 1 - np.sum(np.square(U_lf_test - U_pred[1][:,0])) / np.sum(np.square(U_lf_test - np.mean(U_lf_test)))
        r2_LF_list.append(r2_LF)
        print(f"R^2 LF: {r2_LF:.4f}")

        print('\n \n')

    r2_LF_df[str(Nlf_models[m])] = r2_LF_list
    r2_HF_df[str(Nlf_models[m])] = r2_HF_list
    r2_df[str(Nlf_models[m])] = r2_list
    mse_LF_df[str(Nlf_models[m])] = test_mse_LF_list
    mse_HF_df[str(Nlf_models[m])] = test_mse_HF_list
    mse_df[str(Nlf_models[m])] = test_mse_list
    

print(r2_HF_df.round(5))
print(mse_HF_df.round(5))


********************  #basis functions = 1  ********************
-------  #HF data = 5  -------


4/4 [==============================] - 0s 2ms/step
Elapsed time:  42.12251549999928
Test MSE: 0.00001865
Test MSE HF: 0.00003721
Test MSE LF: 0.00000008
R^2: 0.4393
R^2 HF: -0.1160
R^2 LF: 0.9976

 

-------  #HF data = 7  -------
4/4 [==============================] - 0s 3ms/step
Elapsed time:  39.7916125000047
Test MSE: 0.00001964
Test MSE HF: 0.00003921
Test MSE LF: 0.00000008
R^2: 0.4093
R^2 HF: -0.1758
R^2 LF: 0.9976

 

-------  #HF data = 9  -------
4/4 [==============================] - 0s 0s/step
Elapsed time:  39.446023700002115
Test MSE: 0.00001848
Test MSE HF: 0.00003689
Test MSE LF: 0.00000007
R^2: 0.4444
R^2 HF: -0.1062
R^2 LF: 0.9978

 

-------  #HF data = 11  -------
4/4 [==============================] - 0s 3ms/step
Elapsed time:  39.61788330000127
Test MSE: 0.00001693
Test MSE HF: 0.00003379
Test MSE LF: 0.00000007
R^2: 0.4909
R^2 HF: -0.0133
R^2 LF: 0.9977

 

-------  #HF data = 13  -------
4/4 [==============================] - 0s 3ms/step
Elapsed time:  40.146548

In [7]:
#########################     SAVE the OUTPUT      ##########################
os.makedirs('Output_new_Lin')

r2_HF_df.to_csv('./Output_new_Lin/r2_HF_lhs.txt', header=True, index=False, sep='\t', mode='a')
mse_HF_df.to_csv('./Output_new_Lin/mse_HF_lhs.txt', header = True, index = False, sep = '\t', mode = 'a')
r2_LF_df.to_csv('./Output_new_Lin/r2_LF_lhs.txt', header=True, index=False, sep='\t', mode='a')
mse_LF_df.to_csv('./Output_new_Lin/mse_LF_lhs.txt', header = True, index = False, sep = '\t', mode = 'a')


with open('./Output_new_Lin/U_HF_list.data', 'wb') as filehandle:
    # store the data as binary data stream
    pickle.dump(U_HF_list, filehandle)

with open('./Output_new_Lin/U_LF_list.data', 'wb') as filehandle:
    pickle.dump(U_LF_list, filehandle)
